### Parsing Accuracy Evaluation (WER & CER)

##### This code compares the OCR output with the ground-truth text using Word Error Rate (WER) and Character Error Rate (CER) from torchmetrics.text, and then converts the error rates into accuracy scores. Both WER and CER measure how many edits (substitutions, deletions, insertions) are required to transform the OCR text into the correct text.

#### Formula:
#### Error Rate = (S+D+I)/N​
##### Where S = substitutions, D = deletions, I = insertions, and N = total words (for WER) or total characters (for CER).

In [2]:
from torchmetrics.text import WordErrorRate, CharErrorRate
import os

# Define file paths
ocr_output_path = "../data/output_text_Cr.No.01-2017.txt"
truth_data_path = "../data/truth_data_Cr.No.01-2017.txt"

# Check if files exist
if not os.path.exists(ocr_output_path):
    print(f"ERROR: OCR output file not found: {ocr_output_path}")
elif not os.path.exists(truth_data_path):
    print(f"ERROR: Truth data file not found: {truth_data_path}")
else:
    # Load OCR output (predictions)
    with open(ocr_output_path, "r", encoding="utf-8") as f:
        ocr_output = f.read().strip()
    
    # Load Truth data (ground truth)
    with open(truth_data_path, "r", encoding="utf-8") as f:
        truth_data = f.read().strip()
    
    # Initialize metrics
    wer_metric = WordErrorRate()
    cer_metric = CharErrorRate()
    
    # Compute error rates
    # WordErrorRate and CharErrorRate expect strings and compute edit distances
    wer_score = wer_metric(ocr_output, truth_data)
    cer_score = cer_metric(ocr_output, truth_data)
    
    # Convert to accuracy scores
    word_accuracy = 1 - wer_score
    char_accuracy = 1 - cer_score
    
    # Display results
    print("PARSING EVALUATION RESULTS")
    print(f"Word Error Rate (WER):     {wer_score:.4f}")
    print(f"Character Error Rate (CER): {cer_score:.4f}")
    print(f"Word Accuracy:             {word_accuracy:.4f} ({word_accuracy*100:.2f}%)")
    print(f"Character Accuracy:        {char_accuracy:.4f} ({char_accuracy*100:.2f}%)")

PARSING EVALUATION RESULTS
Word Error Rate (WER):     0.3100
Character Error Rate (CER): 0.1530
Word Accuracy:             0.6900 (69.00%)
Character Accuracy:        0.8470 (84.70%)


### Detailed Text Comparison and Statistics

##### This cell provides detailed statistics about the OCR output vs ground truth, including word/character counts, and a side-by-side comparison of the texts.

In [5]:
# Load and analyze OCR output
with open("../data/output_text_Cr.No.01-2017.txt", "r", encoding="utf-8") as f:
    ocr_text = f.read()

ocr_word_count = len(ocr_text.split())
ocr_char_count = len(ocr_text)

print("OCR OUTPUT STATISTICS")
print(f"Total characters: {ocr_char_count}")
print(f"Total words:      {ocr_word_count}")

OCR OUTPUT STATISTICS
Total characters: 9509
Total words:      1088


In [10]:
with open("../data/truth_data_Cr.No.01-2017.txt", "r", encoding="utf-8") as f:
    text = f.read()
word_count = len(text.split())
char_count = len(text)

print("\nTRUTH DATA STATISTICS")
print("Total characters:", char_count)
print("Total words:", word_count)


TRUTH DATA STATISTICS
Total characters: 9826
Total words: 1171


### Detailed Error Analysis

##### This cell performs a detailed comparison between OCR and ground truth text, showing first differences and example mismatches.

In [11]:
# Detailed comparison between OCR and ground truth
import difflib

with open("../data/output_text_Cr.No.01-2017.txt", "r", encoding="utf-8") as f:
    ocr_text = f.read()

with open("../data/truth_data_Cr.No.01-2017.txt", "r", encoding="utf-8") as f:
    truth_text = f.read()

# Split into words for detailed comparison
ocr_words = ocr_text.split()
truth_words = truth_text.split()

# Word-level comparison
matcher = difflib.SequenceMatcher(None, ocr_words, truth_words)
matching_blocks = matcher.get_matching_blocks()

total_matching_words = sum(block.size for block in matching_blocks)
word_match_percentage = (total_matching_words / len(truth_words) * 100) if truth_words else 0

print("DETAILED WORD-LEVEL ANALYSIS")
print(f"Truth words:       {len(truth_words)}")
print(f"OCR words:         {len(ocr_words)}")
print(f"Matching words:    {total_matching_words}")
print(f"Word Match %:      {word_match_percentage:.2f}%")

# Show first differences
ocr_lines = ocr_text.split('\n')
truth_lines = truth_text.split('\n')

print("\nFIRST 3 LINES COMPARISON:")
for i in range(min(3, len(truth_lines))):
    print(f"\n[Line {i+1}]")
    print(f"Truth: {truth_lines[i][:80]}...")
    print(f"OCR:   {ocr_lines[i][:80] if i < len(ocr_lines) else 'MISSING'}...")
    if i < len(ocr_lines) and truth_lines[i] != ocr_lines[i]:
        print(f"Mismatch")

DETAILED WORD-LEVEL ANALYSIS
Truth words:       1171
OCR words:         1088
Matching words:    854
Word Match %:      72.93%

FIRST 3 LINES COMPARISON:

[Line 1]
Truth: FIRSTINFORMATION REPORT TAMILNADUPOLICE முதல் தகவல் அறிக்கை...
OCR:   FIRSTINFORMATION REPORT TAMILNADUPOLICE முதல் தகவல் அறிக்கை...

[Line 2]
Truth: INTEGRATED INVESTIGATIONAL FORM-I (UnderSection154Cr.P.C) C 7405314 கு.ந.வி.தொ.ப...
OCR:   INTEGRATEDINVESTIGATIONAL (UnderSection154Cr.P.C) 7405314 குநவிதொயிரிவு 154 இன்...
Mismatch

[Line 3]
Truth: FIRNo: 1 Date: 06-04-2017 ...
OCR:   கீது C 2017 PSI SALEM Year:...
Mismatch
